# Model Evaluation

The following notebook proposes to study how the model behaves on the test set *before* and *after* the finetuning operations takes part.


The Notebook is divided in two parts:
- A first approach part where the initial model is loaded tested and run on some examples.
- Secondly the model is ran and the results are saved, this more advanced and curated part is where we also run the inference for the fine tuned model.

### Imports

In [2]:
from pathlib import Path
from datasets import load_from_disk
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import subprocess
import transformers, sys
import shutil
import time
import json
from peft import PeftModel


/mnt/data/envs/finetune-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading the Datasets

First we define the absolute paths and secondly we get the data.

In [3]:
PROJECT_ROOT = Path("/mnt/data/projects/Multilingual-Finetuning-Mistral-3B").resolve()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

ARTIFACTS_ROOT = Path("/mnt/data/project_artifacts")    # We store all the model cache, offload files, outputs and logs in the external volume
MODEL_CACHE = ARTIFACTS_ROOT / "models_cache"
OFFLOAD_DIR = ARTIFACTS_ROOT / "offload"
OUTPUTS_DIR = ARTIFACTS_ROOT / "outputs"
LOGS_DIR = ARTIFACTS_ROOT / "logs"
CHECKPOINT_DIR = ARTIFACTS_ROOT / "checkpoints"

for p in [MODEL_CACHE, OFFLOAD_DIR, OUTPUTS_DIR, LOGS_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)
print("MODEL_CACHE:", MODEL_CACHE)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)


PROJECT_ROOT: /mnt/data/projects/Multilingual-Finetuning-Mistral-3B
DATA_PROCESSED: /mnt/data/projects/Multilingual-Finetuning-Mistral-3B/data/processed
ARTIFACTS_ROOT: /mnt/data/project_artifacts
MODEL_CACHE: /mnt/data/project_artifacts/models_cache
CHECKPOINT_DIR: /mnt/data/project_artifacts/checkpoints


In [4]:
test_ds = load_from_disk(dataset_path= DATA_PROCESSED / "test")
control_por = load_from_disk(dataset_path= DATA_PROCESSED / "control_por")
control_spa = load_from_disk(dataset_path= DATA_PROCESSED / "control_spa")


### Loading the Model

This part is related to the initial load of the not finetuned model. The model is initially defined and loaded.

First we dowload the tokenizer, which is the responsable for converting the text from words to the numerical form used by the model, and viceversa. (it takes some MB of storage)

Secondly we load the model. This is the actual brain, with billions of parameters, its role is to predict the tokens. (constitutes the expensive part for what concerns download time, disk space and RAM usage)

Note: even though they are loaded independently they both are associated to the same model, since the model is trained expecting that exact token scheme and properly works only with that one.

The following functions will be used both to run over the baseline model and the finetuned one. To do so we define a run label that once chosen helps define the type of computations currently being runned and automatically defines new folders where to store the results.


In [6]:
RUN_LABEL = "baseline" 
#RUN_LABEL = "finetuned" 


In [7]:
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

FINETUNED_RUN_NAME = "qwen25_3b_multilingual_lora_actual_run1"
FINETUNED_RUN_DIR = CHECKPOINT_DIR / FINETUNED_RUN_NAME
FINETUNED_ADAPTER_DIR = FINETUNED_RUN_DIR / "checkpoint-611"

print("RUN_LABEL:", RUN_LABEL)
print("Adapter dir:", FINETUNED_ADAPTER_DIR)
print("Adapter exists:", FINETUNED_ADAPTER_DIR.exists())


RUN_LABEL: baseline
Adapter dir: /mnt/data/project_artifacts/checkpoints/qwen25_3b_multilingual_lora_actual_run1/checkpoint-611
Adapter exists: True


In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    cache_dir=str(MODEL_CACHE),
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


The loading is versatile in order to be able to switch from systems with CPU to GPU ones more efficiently.

In [9]:
def load_base_model(model_name, model_cache):
    if torch.cuda.is_available():
        chosen_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            cache_dir=str(model_cache),
            torch_dtype=chosen_dtype,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            cache_dir=str(model_cache),
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )

    model.eval()
    return model


In [10]:
def load_finetuned_model(model_name, model_cache, adapter_dir):
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Fine-tuned adapter not found: {adapter_dir}")

    base_model = load_base_model(model_name, model_cache)

    model = PeftModel.from_pretrained(
        base_model,
        str(adapter_dir)
    )

    model.eval()
    return model


In [11]:
if RUN_LABEL == "baseline":
    model = load_base_model(BASE_MODEL_NAME, MODEL_CACHE)
    print("Loaded baseline model.")

elif RUN_LABEL == "finetuned" and not FINETUNED_ADAPTER_DIR.exists():
    raise RuntimeError(
        f"RUN_LABEL is 'finetuned' but adapter path does not exist: {FINETUNED_ADAPTER_DIR}"
    )

elif RUN_LABEL == "finetuned":
    model = load_finetuned_model(BASE_MODEL_NAME, MODEL_CACHE, FINETUNED_ADAPTER_DIR)
    print("Loaded fine-tuned model.")

print("RUN_LABEL:", RUN_LABEL)
print("Base model:", BASE_MODEL_NAME)
if RUN_LABEL == "finetuned":
    print("Adapter path:", FINETUNED_ADAPTER_DIR)

print(type(model))
print(type(tokenizer))


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 281.65it/s]


Loaded baseline model.
RUN_LABEL: baseline
Base model: Qwen/Qwen2.5-3B-Instruct
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>


In [12]:
model.eval()

print("model.device:", model.device)

if hasattr(model, "hf_device_map"):
    print("model hf_device_map:", model.hf_device_map)

base = model.get_base_model() if hasattr(model, "get_base_model") else model

if hasattr(base, "hf_device_map"):
    print("base hf_device_map:", base.hf_device_map)

print("input embeddings device:", model.get_input_embeddings().weight.device)


model.device: cuda:0
input embeddings device: cuda:0


In [13]:
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi


CUDA available: True
Mon Apr 27 14:07:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:06.0 Off |                    0 |
| N/A   30C    P0             25W /   70W |    6063MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

#### Verifying that the download was succesful

The following cells are some sanity checks to verify the absence of bugs.

In [14]:
print("MODEL CACHE EXISTS:", MODEL_CACHE.exists())
print("OFFLOAD EXISTS:", OFFLOAD_DIR.exists())

subprocess.run(["du", "-sh", str(MODEL_CACHE)])
subprocess.run(["du", "-sh", str(OFFLOAD_DIR)])


MODEL CACHE EXISTS: True
OFFLOAD EXISTS: True
9.4G	/mnt/data/project_artifacts/models_cache
4.0K	/mnt/data/project_artifacts/offload


CompletedProcess(args=['du', '-sh', '/mnt/data/project_artifacts/offload'], returncode=0)

In [15]:
files = []
for folder in [MODEL_CACHE, OFFLOAD_DIR]:
    if folder.exists():
        for path in folder.rglob("*"):
            if path.is_file():
                files.append((path.stat().st_size, path))

files = sorted(files, reverse=True)

for size, path in files[:20]:
    print(f"{size / 1024**2:.2f} MB  {path}")


3784.81 MB  /mnt/data/project_artifacts/models_cache/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1/model-00001-of-00002.safetensors
3784.81 MB  /mnt/data/project_artifacts/models_cache/models--Qwen--Qwen2.5-3B-Instruct/blobs/67347b23fb4165b652eb6611f5e1f2a06dfcddba8e909df1b2b0b1857bee06c2
3654.38 MB  /mnt/data/project_artifacts/models_cache/models--Aratako--Ministral-3-3B-Instruct-2512-TextOnly/snapshots/e5c347cdff40c5e53d23e6230b6504bfa18f6a3d/model.safetensors
3654.38 MB  /mnt/data/project_artifacts/models_cache/models--Aratako--Ministral-3-3B-Instruct-2512-TextOnly/blobs/47d7ef2f8bdeb57c653de1833a517f5d565e6f339db832a31fbeffc07b8c59e0
2101.20 MB  /mnt/data/project_artifacts/models_cache/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1/model-00002-of-00002.safetensors
2101.20 MB  /mnt/data/project_artifacts/models_cache/models--Qwen--Qwen2.5-3B-Instruct/blobs/a40d941d0e7e0b966ad8b62bb6d6b7c88cce1299197b599d9d0

### Testing the Dataset
We begin by inspecting the datasets and verifying the keys and column names. Then we run the model for a simple example run to later expand to an iterative inference.

In [16]:
example = test_ds[302]
print(test_ds[0],"\n", test_ds.column_names, "\n",test_ds.shape)

print("Language code:", example["language_code"])
print("\nINPUT:\n", example["inputs"])
print("\nTARGET:\n", example["targets"])


{'inputs': 'What is the significance of time-series analysis in data science, and where is it commonly applied?', 'targets': 'Time-series analysis is crucial for understanding temporal patterns in data. It is commonly applied in finance, forecasting, and IoT applications to analyze trends over time.', 'language': 'English', 'language_code': 'eng', 'annotation_type': 'original-annotations', 'user_id': '7e2f92b1fdb1a83cbd6d507fff9c5478fee7da855370d4644984399159bbf852'} 
 ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id'] 
 (612, 6)
Language code: eng

INPUT:
 What challenges arise in handling low-resource languages in NLP?

TARGET:
 Low-resource languages pose challenges due to limited annotated data. Techniques like transfer learning and cross-lingual approaches help address these challenges.


Since we are working with a chat-tuned model it expects the inputs in a conversation format and not simple plain text so we proceed by creating the chat-style prompt to later do inference.

The following step is to appropriately create the prompt text by means of the tokenizer, which converts the message to the exact input exected by the model.

Once the message is correctly formatted we tokenize it.

In [17]:
messages = [
    {"role": "user", "content": example["inputs"]}
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(prompt_text)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}


<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What challenges arise in handling low-resource languages in NLP?<|im_end|>
<|im_start|>assistant



#### Generating an answer

This is the first point where the implemented model generates an actual answers given an arbitrary input by performing inference.
We set the limit to 64 tokens to balance exploration and answer length.

We then proceed to decode and print the newly generated part (prediction) and compare with the target.

In [18]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,    # This makes the generation deterministic
        pad_token_id=tokenizer.eos_token_id,
    )


prompt_len = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][prompt_len:]

prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)


print("=== TARGET ===")
print(example["targets"])

print("\n=== PREDICTION ===")
print(prediction)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== TARGET ===
Low-resource languages pose challenges due to limited annotated data. Techniques like transfer learning and cross-lingual approaches help address these challenges.

=== PREDICTION ===
Handling low-resource languages in Natural Language Processing (NLP) presents several unique challenges that differ from those faced with more widely used languages. Here are some of the key challenges:

1. **Data Availability**: Low-resource languages often have limited amounts of annotated data available for training NLP models. This scarcity can severely limit the


#### Time efficiency analysis

The following cell replicates what we just did and is used to study how much time each part of the training takes and to understand the order of magnitude of the time that the whole inference computation will take.

Indicatively 8/9s of run time is related to a model that is running on CPU. Meanwhile a model on GPU takes approximately 2/3s.

In [19]:
start = time.time()

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

mid1 = time.time()

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

mid2 = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

mid3 = time.time()

prompt_len = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][prompt_len:]
prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)

end = time.time()

print("chat template:", mid1 - start)
print("tokenization:", mid2 - mid1)
print("generation:", mid3 - mid2)
print("decode:", end - mid3)
print("total:", end - start)


chat template: 0.00015783309936523438
tokenization: 0.0012760162353515625
generation: 2.9616236686706543
decode: 0.0003218650817871094
total: 2.963379383087158


### Run Over the whole dataset

The following functions are defined to iterate the just shown process over the whole datasets. 

A key component is the "load_done_indeces" function, which keeps track of the computations already done and allows to resume the inference from the point it was previously stopped.


In [20]:
def get_inference_device(model):    # Allows to switch from CPU to GPU without bugs
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


In [21]:
def generate_prediction(example, model, tokenizer, max_new_tokens=64):
    messages = [
        {"role": "user", "content": example["inputs"]}
    ]
    
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    )
    
    device = get_inference_device(model)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][prompt_len:]
    prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)   # turns the generated tokens IDs back to normal text
    
    return prediction


In [22]:
def load_done_indices(output_file):
    done = set()
    
    if output_file.exists():
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                done.add(row["index"])
    
    return done


The following function merges everything, it runs the inference on the whole dataset and saves the results incrementally to the file.

In [23]:
def run_dataset_eval(ds, ds_name, output_file, model, tokenizer, run_label, max_new_tokens=64):
    done_indices = load_done_indices(output_file)   # loads the already done indices to avoid recomputing already done computations
    
    print(f"{ds_name}: already done {len(done_indices)} examples")
    
    with open(output_file, "a", encoding="utf-8") as f: # opend the output file in append mode "a"
        for i, example in enumerate(ds):
            if i in done_indices:
                continue    # skips exaamples already computed
            
            prediction = generate_prediction(   # runs the model by calling the function
                example,
                model,
                tokenizer,
                max_new_tokens=max_new_tokens,
            )
            
            row = {      # this part builds the results dataset, by storing the relavant information of the particular inference run just done
                "run_label": run_label,
                "dataset": ds_name,
                "index": i,
                "language_code": example["language_code"],
                "input": example["inputs"],
                "target": example["targets"],
                "prediction": prediction,
            }
            
            f.write(json.dumps(row, ensure_ascii=False) + "\n") # writes the results on a single line on the file
            f.flush()   # this forces the write to be saved immediately
            
            if i % 10 == 0:
                print(f"{ds_name}: completed {i+1}/{len(ds)}")  # completeness progress tracking


### Saving the Results

Having defined the functions that will be used to run the inference over the whole dataset, we continue by defining the paths where to store the results of the models.

In [24]:
OUTPUTS_DIR = Path("/mnt/data/project_artifacts/outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

test_output = OUTPUTS_DIR / f"{RUN_LABEL}_test.jsonl"
spa_output = OUTPUTS_DIR / f"{RUN_LABEL}_control_spa.jsonl"
por_output = OUTPUTS_DIR / f"{RUN_LABEL}_control_por.jsonl"


Test run to verify the correct behaviour of the functions over a restricted dataset

In [25]:
small_test = test_ds.select(range(20))

run_dataset_eval(
    small_test,
    "small_test",
    test_output,
    model,
    tokenizer,
    RUN_LABEL,
    max_new_tokens=64,
)


small_test: already done 0 examples
small_test: completed 1/20
small_test: completed 11/20


Running the inference over the whole datasets.

In [26]:
run_dataset_eval(
    test_ds,
    f"{RUN_LABEL}_test",
    test_output,
    model,
    tokenizer,
    RUN_LABEL,
    max_new_tokens=64,
)


run_dataset_eval(
    control_spa,
    f"{RUN_LABEL}_control_spa",
    spa_output,
    model,
    tokenizer,
    RUN_LABEL,
    max_new_tokens=64,
)

run_dataset_eval(
    control_por,
    f"{RUN_LABEL}_control_por",
    por_output,
    model,
    tokenizer,
    RUN_LABEL,
    max_new_tokens=64,
)


baseline_test: already done 20 examples
baseline_test: completed 21/612
baseline_test: completed 31/612
baseline_test: completed 41/612
baseline_test: completed 51/612
baseline_test: completed 61/612
baseline_test: completed 71/612
baseline_test: completed 81/612
baseline_test: completed 91/612
baseline_test: completed 101/612
baseline_test: completed 111/612
baseline_test: completed 121/612
baseline_test: completed 131/612
baseline_test: completed 141/612
baseline_test: completed 151/612
baseline_test: completed 161/612
baseline_test: completed 171/612
baseline_test: completed 181/612
baseline_test: completed 191/612
baseline_test: completed 201/612
baseline_test: completed 211/612
baseline_test: completed 221/612
baseline_test: completed 231/612
baseline_test: completed 241/612
baseline_test: completed 251/612
baseline_test: completed 261/612
baseline_test: completed 271/612
baseline_test: completed 281/612
baseline_test: completed 291/612
baseline_test: completed 301/612
baseline_te

In [30]:
for file, ds_name, ds in [
    (test_output, f"{RUN_LABEL}_test", test_ds),
    (spa_output, f"{RUN_LABEL}_control_spa", control_spa),
    (por_output, f"{RUN_LABEL}_control_por", control_por),
]:
    indices = []

    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            indices.append(row["index"])

    print(f"\n===== {ds_name} =====")
    print("First few indices:", indices[:10])
    print("Last few indices:", indices[-10:])
    print("Unique indices:", len(set(indices)))
    print("Total rows:", len(indices))

    expected = set(range(len(ds)))
    found = set(indices)

    print("Missing:", sorted(expected - found)[:10])
    print("Extra:", sorted(found - expected)[:10])



===== baseline_test =====
First few indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Last few indices: [602, 603, 604, 605, 606, 607, 608, 609, 610, 611]
Unique indices: 612
Total rows: 612
Missing: []
Extra: []

===== baseline_control_spa =====
First few indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Last few indices: [290, 291, 292, 293, 294, 295, 296, 297, 298, 299]
Unique indices: 300
Total rows: 300
Missing: []
Extra: []

===== baseline_control_por =====
First few indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Last few indices: [290, 291, 292, 293, 294, 295, 296, 297, 298, 299]
Unique indices: 300
Total rows: 300
Missing: []
Extra: []
